# NLP Lab – Unit 5: Subword Tokenization, POS Tagging & Word2Vec

| Section | Topic |
|---|---|
| 0 | Setup & data loading |
| 5.1 | BPE tokenization (a. pretrained, b. trained from scratch) |
| 5.2 | SentencePiece tokenization (a. pretrained, b. trained from scratch) |
| 5.3 | POS tagging on a sentence (a. spaCy, b. NLTK) |
| 5.4 | POS tagging with frequency on the input file (spaCy) |
| 5.5 | Word2Vec – CBOW |
| 5.6 | Word2Vec – Skip-Gram |
| 5.7 | Cosine similarity between word pairs |

## 0. Setup & Data Loading

### 0.1 Install dependencies

In [1]:
!pip install -q tokenizers transformers sentencepiece spacy nltk gensim pandas gdown
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 35.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 70.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### 0.2 Download the input file (only if it is not already present)

In [2]:
import os
import gdown

INPUT_PATH = "input_sub_word_data.txt"
FILE_ID = "1eHA0Q9ju-08n_hByCK03GqWAX9HQvAKd"

if not os.path.exists(INPUT_PATH):
    gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", INPUT_PATH, quiet=False)

print("File exists:", os.path.exists(INPUT_PATH))

Downloading...
From: https://drive.google.com/uc?id=1eHA0Q9ju-08n_hByCK03GqWAX9HQvAKd
To: /content/input_sub_word_data.txt
100%|██████████| 6.80k/6.80k [00:00<00:00, 14.1MB/s]

File exists: True


### 0.3 Load the data and split into sentences

In [3]:
import re
import pandas as pd

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    text = f.read().strip()

# Split on sentence-ending punctuation or newlines
sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n+', text) if s.strip()]

print("Characters :", len(text))
print("Sentences  :", len(sentences))

# Number of sentences to display in the tokenization outputs (set to len(sentences) for all)
N_SHOW = 5
demo_sentences = sentences[:N_SHOW]

for s in demo_sentences:
    print("-", s)

Characters : 6682
Sentences  : 128
- Natural language processing is a branch of artificial intelligence that
- enables computers to understand, interpret, and generate human language.
- Natural language processing combines linguistics, computer science, and
- machine learning to process large amounts of natural language data.
- Tokenization is one of the fundamental steps in natural language processing.


### 0.4 Helper to display tokens and IDs

In [4]:
def show_tokens(sentence, tokens, ids):
    """Print a sentence with its tokens and token IDs as a table."""
    print("=" * 80)
    print("Sentence :", sentence)
    print("Tokens   :", tokens)
    print("Token IDs:", ids)
    print(pd.DataFrame({"Token": tokens, "Token ID": ids}).to_string(index=False))

---
## 5.1 Subword Tokenization using BPE (Byte Pair Encoding)

### 5.1 (a) Using a pretrained model
GPT-2 uses a byte-level BPE tokenizer. `Ġ` marks a token that starts with a space.

In [5]:
from transformers import AutoTokenizer

bpe_pretrained = AutoTokenizer.from_pretrained("gpt2")
print("Vocabulary size:", bpe_pretrained.vocab_size)

for sent in demo_sentences:
    tokens = bpe_pretrained.tokenize(sent)
    ids = bpe_pretrained.convert_tokens_to_ids(tokens)
    show_tokens(sent, tokens, ids)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Vocabulary size: 50257
Sentence : Natural language processing is a branch of artificial intelligence that
Tokens   : ['Natural', 'Ġlanguage', 'Ġprocessing', 'Ġis', 'Ġa', 'Ġbranch', 'Ġof', 'Ġartificial', 'Ġintelligence', 'Ġthat']
Token IDs: [35364, 3303, 7587, 318, 257, 8478, 286, 11666, 4430, 326]
        Token  Token ID
      Natural     35364
    Ġlanguage      3303
  Ġprocessing      7587
          Ġis       318
           Ġa       257
      Ġbranch      8478
          Ġof       286
  Ġartificial     11666
Ġintelligence      4430
        Ġthat       326
Sentence : enables computers to understand, interpret, and generate human language.
Tokens   : ['en', 'ables', 'Ġcomputers', 'Ġto', 'Ġunderstand', ',', 'Ġinterpret', ',', 'Ġand', 'Ġgenerate', 'Ġhuman', 'Ġlanguage', '.']
Token IDs: [268, 2977, 9061, 284, 1833, 11, 6179, 11, 290, 7716, 1692, 3303, 13]
      Token  Token ID
         en       268
      ables      2977
 Ġcomputers      9061
        Ġto       284
Ġunderstand      1833
    

### 5.1 (b) Without using a pretrained model
A BPE tokenizer is trained from scratch on the input file (Hugging Face `tokenizers`).
`##` marks a subword that continues the previous piece of the same word.

In [6]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

bpe_scratch = Tokenizer(models.BPE(unk_token="[UNK]", continuing_subword_prefix="##"))
bpe_scratch.pre_tokenizer = pre_tokenizers.Whitespace()

bpe_trainer = trainers.BpeTrainer(
    vocab_size=1000,                       # upper limit; a small file will produce fewer
    special_tokens=["[UNK]", "[PAD]"],
    continuing_subword_prefix="##",
)
bpe_scratch.train([INPUT_PATH], bpe_trainer)
print("Learned vocabulary size:", bpe_scratch.get_vocab_size())

for sent in demo_sentences:
    enc = bpe_scratch.encode(sent)
    show_tokens(sent, enc.tokens, enc.ids)

Learned vocabulary size: 1000
Sentence : Natural language processing is a branch of artificial intelligence that
Tokens   : ['Natural', 'language', 'processing', 'is', 'a', 'branch', 'of', 'artific', '##ial', 'intellig', '##ence', 'that']
Token IDs: [409, 175, 227, 136, 21, 970, 114, 906, 456, 851, 204, 311]
     Token  Token ID
   Natural       409
  language       175
processing       227
        is       136
         a        21
    branch       970
        of       114
   artific       906
     ##ial       456
  intellig       851
    ##ence       204
      that       311
Sentence : enables computers to understand, interpret, and generate human language.
Tokens   : ['enables', 'computers', 'to', 'understand', ',', 'inter', '##pre', '##t', ',', 'and', 'generate', 'human', 'language', '.']
Token IDs: [913, 936, 90, 258, 2, 790, 736, 50, 2, 102, 904, 989, 175, 4]
     Token  Token ID
   enables       913
 computers       936
        to        90
understand       258
         ,        

---
## 5.2 Subword Tokenization using SentencePiece

### 5.2 (a) Using a pretrained model
The T5 tokenizer is a pretrained SentencePiece model. `▁` marks the beginning of a word.

In [7]:
from transformers import AutoTokenizer

sp_pretrained = AutoTokenizer.from_pretrained("t5-small", use_fast=False)
print("Vocabulary size:", sp_pretrained.vocab_size)

for sent in demo_sentences:
    tokens = sp_pretrained.tokenize(sent)
    ids = sp_pretrained.convert_tokens_to_ids(tokens)
    show_tokens(sent, tokens, ids)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Vocabulary size: 32100
Sentence : Natural language processing is a branch of artificial intelligence that
Tokens   : ['▁Natural', '▁language', '▁processing', '▁is', '▁', 'a', '▁branch', '▁of', '▁artificial', '▁intelligence', '▁that']
Token IDs: [6869, 1612, 3026, 19, 3, 9, 6421, 13, 7353, 6123, 24]
        Token  Token ID
     ▁Natural      6869
    ▁language      1612
  ▁processing      3026
          ▁is        19
            ▁         3
            a         9
      ▁branch      6421
          ▁of        13
  ▁artificial      7353
▁intelligence      6123
        ▁that        24
Sentence : enables computers to understand, interpret, and generate human language.
Tokens   : ['▁', 'enables', '▁computers', '▁to', '▁understand', ',', '▁interpret', ',', '▁and', '▁generate', '▁human', '▁language', '.']
Token IDs: [3, 7161, 7827, 12, 734, 6, 7280, 6, 11, 3806, 936, 1612, 5]
      Token  Token ID
          ▁         3
    enables      7161
 ▁computers      7827
        ▁to        12
▁understa

### 5.2 (b) Without using a pretrained model
A SentencePiece model is trained from scratch on the same input file.

In [8]:
import sentencepiece as spm

# SentencePiece expects one sentence per line
with open("sp_train.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(sentences))

spm.SentencePieceTrainer.train(
    input="sp_train.txt",
    model_prefix="sp_model",
    vocab_size=500,
    model_type="unigram",          # can also be "bpe", "char" or "word"
    hard_vocab_limit=False,        # lets the trainer shrink the vocab for small files
)

sp_scratch = spm.SentencePieceProcessor(model_file="sp_model.model")
print("Learned vocabulary size:", sp_scratch.get_piece_size())

for sent in demo_sentences:
    tokens = sp_scratch.encode(sent, out_type=str)
    ids = sp_scratch.encode(sent, out_type=int)
    show_tokens(sent, tokens, ids)

Learned vocabulary size: 494
Sentence : Natural language processing is a branch of artificial intelligence that
Tokens   : ['▁Natural', '▁language', '▁processing', '▁is', '▁a', '▁b', 'ran', 'ch', '▁of', '▁arti', 'fic', 'ial', '▁inte', 'll', 'ig', 'ence', '▁that']
Token IDs: [177, 19, 39, 16, 9, 292, 406, 146, 11, 317, 143, 423, 330, 273, 402, 156, 78]
      Token  Token ID
   ▁Natural       177
  ▁language        19
▁processing        39
        ▁is        16
         ▁a         9
         ▁b       292
        ran       406
         ch       146
        ▁of        11
      ▁arti       317
        fic       143
        ial       423
      ▁inte       330
         ll       273
         ig       402
       ence       156
      ▁that        78
Sentence : enables computers to understand, interpret, and generate human language.
Tokens   : ['▁en', 'able', 's', '▁computer', 's', '▁to', '▁understand', ',', '▁inter', 'pret', ',', '▁and', '▁genera', 'te', '▁h', 'um', 'an', '▁language', '.']
Token

---
## 5.3 POS Tagging on a Given Sentence

In [9]:
sentence_53 = "The young student is reading an interesting book in the library."

### 5.3 (a) Using spaCy

In [10]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp(sentence_53)

seen = set()
rows = []
for token in doc:
    if token.text.lower() in seen:
        continue                      # keep unique tokens only
    seen.add(token.text.lower())
    rows.append({
        "Token": token.text,
        "POS Tag": token.pos_,
        "Description": spacy.explain(token.pos_),
    })

print(pd.DataFrame(rows).to_string(index=False))

      Token POS Tag Description
        The     DET  determiner
      young     ADJ   adjective
    student    NOUN        noun
         is     AUX   auxiliary
    reading    VERB        verb
         an     DET  determiner
interesting     ADJ   adjective
       book    NOUN        noun
         in     ADP  adposition
    library    NOUN        noun
          .   PUNCT punctuation


### 5.3 (b) Using NLTK

In [11]:
import nltk

for pkg in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

# Penn Treebank tag descriptions
PENN_TAGS = {
    "CC": "Coordinating conjunction", "CD": "Cardinal number", "DT": "Determiner",
    "EX": "Existential there", "FW": "Foreign word", "IN": "Preposition / subordinating conjunction",
    "JJ": "Adjective", "JJR": "Adjective, comparative", "JJS": "Adjective, superlative",
    "LS": "List item marker", "MD": "Modal verb", "NN": "Noun, singular or mass",
    "NNS": "Noun, plural", "NNP": "Proper noun, singular", "NNPS": "Proper noun, plural",
    "PDT": "Predeterminer", "POS": "Possessive ending", "PRP": "Personal pronoun",
    "PRP$": "Possessive pronoun", "RB": "Adverb", "RBR": "Adverb, comparative",
    "RBS": "Adverb, superlative", "RP": "Particle", "SYM": "Symbol", "TO": "to",
    "UH": "Interjection", "VB": "Verb, base form", "VBD": "Verb, past tense",
    "VBG": "Verb, gerund / present participle", "VBN": "Verb, past participle",
    "VBP": "Verb, non-3rd person singular present", "VBZ": "Verb, 3rd person singular present",
    "WDT": "Wh-determiner", "WP": "Wh-pronoun", "WP$": "Possessive wh-pronoun",
    "WRB": "Wh-adverb", ".": "Punctuation (sentence end)", ",": "Punctuation (comma)",
    ":": "Punctuation (colon / dash)", "``": "Opening quote", "''": "Closing quote",
}

tokens = nltk.word_tokenize(sentence_53)
tagged = nltk.pos_tag(tokens)

seen = set()
rows = []
for word, tag in tagged:
    if word.lower() in seen:
        continue
    seen.add(word.lower())
    rows.append({"Token": word, "POS Tag": tag, "Description": PENN_TAGS.get(tag, "N/A")})

print(pd.DataFrame(rows).to_string(index=False))

      Token POS Tag                             Description
        The      DT                              Determiner
      young      JJ                               Adjective
    student      NN                  Noun, singular or mass
         is     VBZ       Verb, 3rd person singular present
    reading     VBG       Verb, gerund / present participle
         an      DT                              Determiner
interesting      JJ                               Adjective
       book      NN                  Noun, singular or mass
         in      IN Preposition / subordinating conjunction
    library      NN                  Noun, singular or mass
          .       .              Punctuation (sentence end)


---
## 5.4 POS Tagging with Frequency (spaCy) on the Input File
Tokens are lower-cased so that "The" and "the" are counted together. Whitespace tokens are skipped.

In [12]:
import spacy
from collections import Counter

nlp = spacy.load("en_core_web_sm")
nlp.max_length = max(nlp.max_length, len(text) + 1000)   # safe for long files

doc = nlp(text)

freq = Counter((tok.text.lower(), tok.pos_) for tok in doc if not tok.is_space)

df_54 = pd.DataFrame(
    [{"Token": tok, "POS Tag": pos, "Description": spacy.explain(pos), "Frequency": count}
     for (tok, pos), count in freq.items()]
).sort_values(["Frequency", "Token"], ascending=[False, True]).reset_index(drop=True)

print("Unique (token, POS) pairs:", len(df_54))
df_54.head(30)

Unique (token, POS) pairs: 395


,Token,POS Tag,Description,Frequency
0,",",PUNCT,punctuation,76
1,.,PUNCT,punctuation,65
2,the,DET,determiner,47
3,a,DET,determiner,28
4,and,CCONJ,coordinating conjunction,27
5,can,AUX,auxiliary,23
6,of,ADP,adposition,22
7,be,AUX,auxiliary,15
8,is,AUX,auxiliary,14
9,as,ADP,adposition,13


---
## Corpus for 5.5, 5.6 and 5.7

In [13]:
corpus_text = """Natural language processing is interesting
Natural language processing is useful
I love machine learning
Machine learning is useful
Deep learning is a part of machine learning
Artificial intelligence is changing the world
Natural language processing uses machine learning
Deep learning and artificial intelligence are related
I study natural language processing
Machine learning helps artificial intelligence"""

with open("corpus.txt", "w", encoding="utf-8") as f:
    f.write(corpus_text)

# Read back and tokenize (lower-case, whitespace split)
with open("corpus.txt", "r", encoding="utf-8") as f:
    corpus_sentences = [line.lower().split() for line in f if line.strip()]

print(corpus_sentences[:3])

[['natural', 'language', 'processing', 'is', 'interesting'], ['natural', 'language', 'processing', 'is', 'useful'], ['i', 'love', 'machine', 'learning']]


## 5.5 Word2Vec – CBOW
`sg=0` selects the CBOW architecture.

In [14]:
from gensim.models import Word2Vec

cbow_model = Word2Vec(
    sentences=corpus_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    sg=0,               # 0 = CBOW
    epochs=200,
    seed=42,
    workers=1,          # single worker for reproducible results
)

word = "learning"
print(f"Vector for '{word}':\n", cbow_model.wv[word])

print(f"\nTop 3 words most similar to '{word}':")
for w, score in cbow_model.wv.most_similar(word, topn=3):
    print(f"  {w:<15} {score:.4f}")

Vector for 'learning':
 [-0.01265218  0.01312787  0.00568699  0.00067546 -0.00039841  0.01997441
 -0.02565439  0.01117282 -0.0126041  -0.02041918  0.00884585  0.02831058
  0.00549747  0.01508867  0.01037336  0.02101269  0.00334411 -0.02237608
  0.0118279  -0.0068095  -0.00343869 -0.01332745 -0.01474461  0.01838676
  0.02454898  0.00096197 -0.00397776  0.00756955 -0.00210602 -0.00235707
 -0.00026062 -0.00675901 -0.02064811  0.0071463   0.01516477 -0.01392712
  0.01324037  0.00752126 -0.01507423  0.01093039 -0.00584519  0.01468296
  0.01656358 -0.00365863 -0.01025395  0.01499949  0.00084411  0.01843737
  0.00935418  0.00387894]

Top 3 words most similar to 'learning':
  intelligence    0.3630
  changing        0.2811
  uses            0.2573


## 5.6 Word2Vec – Skip-Gram
`sg=1` selects the Skip-Gram architecture.

In [15]:
skipgram_model = Word2Vec(
    sentences=corpus_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,               # 1 = Skip-Gram
    epochs=200,
    seed=42,
    workers=1,
)

word = "language"
print(f"Vector for '{word}':\n", skipgram_model.wv[word])

print(f"\nTop 5 words most similar to '{word}':")
for w, score in skipgram_model.wv.most_similar(word, topn=5):
    print(f"  {w:<15} {score:.4f}")

Vector for 'language':
 [-3.16276168e-03  1.77079644e-02 -4.04974067e-04  1.06867636e-02
 -6.24552369e-04 -4.58689407e-03  4.37890273e-03  1.98613573e-02
 -1.05280653e-02  8.18031188e-03 -4.27469378e-03  1.53346853e-02
  8.51082988e-03  1.84297597e-03  1.09799253e-02 -2.44846125e-03
 -1.45936627e-02 -2.14204080e-02 -3.50073073e-03  1.22396005e-02
 -1.69535205e-02 -7.32351188e-03  6.85788784e-03 -1.02980714e-02
  1.76095851e-02 -1.08179366e-02  1.22362040e-02  8.45258590e-04
 -1.24796352e-03 -1.26912156e-02 -6.04483706e-04  1.67966988e-02
 -2.17247754e-02  1.30125312e-02 -9.80514451e-05  1.09789493e-02
  5.49309468e-03 -5.84617909e-03 -1.28760329e-02  8.02442990e-03
 -9.62015800e-03  5.90846129e-03 -1.02326619e-02  6.80811470e-03
  9.46135260e-03 -1.88428778e-02  1.24873593e-02 -1.91045832e-03
  1.25400778e-02 -2.30394788e-02]

Top 5 words most similar to 'language':
  intelligence    0.3297
  uses            0.3219
  of              0.2813
  are             0.2801
  is              0.2

## 5.7 Cosine Similarity Between Word Pairs

In [16]:
import numpy as np

sim_model = Word2Vec(
    sentences=corpus_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,
    epochs=200,
    seed=42,
    workers=1,
)

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

pairs = [("machine", "learning"), ("natural", "language"), ("deep", "learning")]

rows = []
for w1, w2 in pairs:
    manual = cosine_similarity(sim_model.wv[w1], sim_model.wv[w2])
    builtin = sim_model.wv.similarity(w1, w2)
    rows.append({"Word 1": w1, "Word 2": w2,
                 "Cosine (manual)": round(float(manual), 4),
                 "Cosine (gensim)": round(float(builtin), 4)})

pd.DataFrame(rows)

,Word 1,Word 2,Cosine (manual),Cosine (gensim)
0,machine,learning,0.3256,0.3256
1,natural,language,0.1505,0.1505
2,deep,learning,0.2539,0.2539


---
**Note:** The corpus in 5.5–5.7 is tiny (10 sentences), so the similarity scores are only illustrative and can change slightly with different seeds or hyperparameters.